# 🧠 DSA Lab 16: Graphs: Representation, BFS, DFS, Topological Sort and Shortest Paths
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 16 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 🚀 Motivation: Everything Is Connected

**A road network is a weighted graph (distances are illustrative)**

```text
  Okara ---- 110 km ---- Lahore
   |   \                 /   |
 60 km  \ 125 km     45 km  50 km
   |     \           /       |
Sahiwal   Faisalabad ---- Sheikhupura
```

## 📖 Part A: Graph Terminology

A **graph** G = (V, E) is a set of **vertices** (nodes) V and a set of **edges** E connecting pairs of vertices.

| Term | Meaning |
|---|---|
| Undirected graph | Edges have no direction (A — B means both ways), e.g., friendship |
| Directed graph (digraph) | Edges have a direction (A → B), e.g., following, prerequisites |
| Weighted graph | Each edge has a cost/weight, e.g., distance, time, price |
| Adjacent / neighbours | Two vertices joined by an edge |
| Degree | Number of edges at a vertex (in-degree / out-degree for directed graphs) |
| Path | A sequence of vertices connected by edges |
| Cycle | A path that starts and ends at the same vertex |
| Connected graph | There is a path between every pair of vertices (undirected) |
| Connected component | A maximal group of vertices that are connected to each other |
| DAG | Directed Acyclic Graph: directed, with no cycles (e.g., course prerequisites) |
| Tree | A connected undirected graph with no cycles (n vertices, n − 1 edges) |

## 🧱 Part B: Representing a Graph

```text
Example (undirected):   0 --- 1
                        |   / |
                        |  /  |
                        2 --- 3      4 (isolated)
Edges: (0,1) (0,2) (1,2) (1,3) (2,3)
```

### Adjacency Matrix

An n × n table where `M[u][v] = 1` (or the weight) if there is an edge u–v. Checking an edge is O(1), but it uses **O(V²)** memory even if there are few edges.

### Adjacency List

For each vertex, store a **list of its neighbours** (in Python, a dictionary of lists). Memory is **O(V + E)**; this is the standard choice for real (sparse) graphs.

In [ ]:
n = 5
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3)]

matrix = [[0] * n for _ in range(n)]
adj = {v: [] for v in range(n)}
for u, v in edges:
    matrix[u][v] = matrix[v][u] = 1          # undirected: both directions
    adj[u].append(v)
    adj[v].append(u)

print("Adjacency matrix:")
for row in matrix:
    print("  ", row)
print("Adjacency list:", adj)
print("degree of 1:", len(adj[1]), "| edge 0-3?", matrix[0][3] == 1)

|  | Adjacency matrix | Adjacency list |
|---|---|---|
| Memory | O(V²) | O(V + E) |
| Check if edge u–v exists | O(1) | O(degree of u) |
| List all neighbours of u | O(V) | O(degree of u) |
| Best for | Dense graphs, small V | Sparse graphs (roads, social networks) |

## 🌊 Part C: Breadth-First Search (BFS)

BFS explores the graph **level by level**: first the start vertex, then all its neighbours, then their neighbours, and so on. It uses a **queue** and a **visited** set. In an unweighted graph, BFS finds the **shortest path (fewest edges)** from the start to every vertex.

**Pseudocode**

```text
BFS(G, s):
    visited ← {s} ; queue ← [s] ; dist[s] ← 0
    WHILE queue not empty:
        u ← dequeue()
        FOR each neighbour v of u:
            IF v not in visited:
                visited.add(v) ; dist[v] ← dist[u] + 1 ; parent[v] ← u
                enqueue(v)
```

In [ ]:
from collections import deque

city = {
    "Okara": ["Sahiwal", "Lahore", "Faisalabad"],
    "Sahiwal": ["Okara", "Multan"],
    "Lahore": ["Okara", "Sheikhupura", "Gujranwala"],
    "Faisalabad": ["Okara", "Sheikhupura"],
    "Sheikhupura": ["Lahore", "Faisalabad"],
    "Gujranwala": ["Lahore"],
    "Multan": ["Sahiwal"],
}

def bfs(graph, start):
    dist, parent = {start: 0}, {start: None}
    q = deque([start])
    while q:
        u = q.popleft()
        for v in graph[u]:
            if v not in dist:              # dist doubles as the visited set
                dist[v] = dist[u] + 1
                parent[v] = u
                q.append(v)
    return dist, parent

def path_to(parent, target):
    path = []
    while target is not None:
        path.append(target)
        target = parent[target]
    return path[::-1]

dist, parent = bfs(city, "Multan")
print(dist)
print("Route Multan → Gujranwala:", " → ".join(path_to(parent, "Gujranwala")))

## 🕳️ Part D: Depth-First Search (DFS)

DFS goes **as deep as possible** along one branch before backtracking, like exploring a maze by always taking the first unexplored corridor. It uses **recursion** (the call stack) or an explicit **stack**.

In [ ]:
def dfs_recursive(graph, u, visited=None, order=None):
    if visited is None:
        visited, order = set(), []
    visited.add(u)
    order.append(u)
    for v in graph[u]:
        if v not in visited:
            dfs_recursive(graph, v, visited, order)
    return order

def dfs_iterative(graph, start):
    visited, order, stack = set(), [], [start]
    while stack:
        u = stack.pop()
        if u in visited:
            continue
        visited.add(u)
        order.append(u)
        for v in reversed(graph[u]):        # reversed: same order as recursive
            if v not in visited:
                stack.append(v)
    return order

print("DFS (recursive):", dfs_recursive(city, "Okara"))
print("DFS (stack)    :", dfs_iterative(city, "Okara"))

|  | BFS | DFS |
|---|---|---|
| Data structure | Queue | Stack / recursion |
| Order | Level by level | Deep first, then backtrack |
| Shortest path (unweighted) | Yes | No |
| Typical uses | Shortest routes, friend suggestions, broadcasting | Cycle detection, topological sort, mazes, components |
| Time | O(V + E) | O(V + E) |

### Connected Components and Cycle Detection

In [ ]:
def components(graph):
    seen, groups = set(), []
    for v in graph:
        if v not in seen:
            group = dfs_recursive(graph, v, seen, [])
            groups.append(sorted(group))
    return groups

def has_cycle_undirected(graph):
    seen = set()
    def dfs(u, parent):
        seen.add(u)
        for v in graph[u]:
            if v not in seen:
                if dfs(v, u):
                    return True
            elif v != parent:               # visited and not the edge we came from
                return True
        return False
    return any(dfs(v, None) for v in graph if v not in seen)

friends = {"Ali": ["Sara"], "Sara": ["Ali", "Hamza"], "Hamza": ["Sara"],
           "Zoya": ["Iqra"], "Iqra": ["Zoya"], "Omar": []}
print("friend groups:", components(friends))
print("cycle in city graph?", has_cycle_undirected(city), "| in friends?", has_cycle_undirected(friends))

## 📋 Part E: Topological Sort (Ordering Tasks)

In a **DAG**, a topological order lists the vertices so that every edge u → v has u **before** v. Example: you must pass Programming Fundamentals before OOP, and OOP before DSA. **Kahn’s algorithm**: repeatedly take a vertex with **in-degree 0** (no remaining prerequisites), output it, and remove its outgoing edges. If some vertices are never output, the graph has a **cycle** (impossible plan).

In [ ]:
def topological_sort(graph):
    indegree = {v: 0 for v in graph}
    for u in graph:
        for v in graph[u]:
            indegree[v] += 1
    q = deque(sorted(v for v in graph if indegree[v] == 0))
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                q.append(v)
    return order if len(order) == len(graph) else None   # None = cycle

courses = {
    "PF": ["OOP", "DLD"], "OOP": ["DSA"], "DLD": ["COAL"], "Discrete": ["DSA"],
    "DSA": ["Algorithms", "DB"], "COAL": ["OS"], "Algorithms": [], "DB": [], "OS": [],
}
print("Study order:", " → ".join(topological_sort(courses)))

## 🛣️ Part F: Dijkstra’s Shortest Path (Weighted Graphs)

When edges have weights (distances, times, costs), BFS no longer gives the shortest path. **Dijkstra’s algorithm** always expands the **closest unexplored vertex**, which it gets from a **min-heap**. It works when all weights are **non-negative**. Time: **O((V + E) log V)**.

**Pseudocode**

```text
Dijkstra(G, s):
    dist[v] ← ∞ for all v ; dist[s] ← 0 ; heap ← [(0, s)]
    WHILE heap not empty:
        (d, u) ← pop smallest
        IF d > dist[u]: CONTINUE                 // outdated entry
        FOR each (v, w) in neighbours of u:
            IF dist[u] + w < dist[v]:            // relaxation
                dist[v] ← dist[u] + w ; parent[v] ← u ; push (dist[v], v)
```

In [ ]:
import heapq

roads = {                                    # km (illustrative)
    "Okara": [("Lahore", 110), ("Sahiwal", 60), ("Faisalabad", 125)],
    "Lahore": [("Okara", 110), ("Sheikhupura", 45)],
    "Sahiwal": [("Okara", 60), ("Faisalabad", 115)],
    "Faisalabad": [("Okara", 125), ("Sahiwal", 115), ("Sheikhupura", 90)],
    "Sheikhupura": [("Lahore", 45), ("Faisalabad", 90)],
}

def dijkstra(graph, source):
    dist = {v: float("inf") for v in graph}
    parent = {source: None}
    dist[source] = 0
    heap = [(0, source)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        for v, w in graph[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                parent[v] = u
                heapq.heappush(heap, (dist[v], v))
    return dist, parent

dist, parent = dijkstra(roads, "Sahiwal")
for city_name, d in sorted(dist.items(), key=lambda kv: kv[1]):
    print(f"{city_name:<12} {d:>4} km   via {' → '.join(path_to(parent, city_name))}")

| Problem | Algorithm | Time |
|---|---|---|
| Visit all vertices / reachability | BFS or DFS | O(V + E) |
| Shortest path, unweighted | BFS | O(V + E) |
| Shortest path, non-negative weights | Dijkstra (heap) | O((V + E) log V) |
| Shortest path with negative weights | Bellman-Ford | O(V · E) |
| Ordering with dependencies | Topological sort | O(V + E) |
| Connect all vertices at minimum cost | Prim / Kruskal (MST) | O(E log V) |

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <vector>
#include <queue>
#include <climits>
using namespace std;

int main() {
    int n = 5;
    vector<vector<pair<int,int>>> adj(n);          // adjacency list: (neighbour, weight)
    auto addEdge = [&](int u, int v, int w) { adj[u].push_back({v, w}); adj[v].push_back({u, w}); };
    addEdge(0, 1, 4); addEdge(0, 2, 1); addEdge(2, 1, 2); addEdge(1, 3, 5); addEdge(3, 4, 3);

    // BFS (ignoring weights)
    vector<int> level(n, -1); queue<int> q; q.push(0); level[0] = 0;
    while (!q.empty()) {
        int u = q.front(); q.pop();
        for (auto [v, w] : adj[u]) if (level[v] == -1) { level[v] = level[u] + 1; q.push(v); }
    }

    // Dijkstra
    vector<int> dist(n, INT_MAX); dist[0] = 0;
    priority_queue<pair<int,int>, vector<pair<int,int>>, greater<>> pq; pq.push({0, 0});
    while (!pq.empty()) {
        auto [d, u] = pq.top(); pq.pop();
        if (d > dist[u]) continue;
        for (auto [v, w] : adj[u])
            if (d + w < dist[v]) { dist[v] = d + w; pq.push({dist[v], v}); }
    }
    for (int i = 0; i < n; i++) cout << i << ": level " << level[i] << ", dist " << dist[i] << endl;
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Degrees and the Class Monitor
In a class of n students (0 … n−1), `trust = [(a, b), ...]` means student a trusts student b (directed). The **monitor** is trusted by everyone else and trusts nobody. Return the monitor’s number or −1.

Hint: compute in-degree and out-degree for every student. The notebook provides `build_graph(n, edges, directed)`.

```text
n = 3, trust = [(0, 2), (1, 2)] → 2
```

In [ ]:
from collections import deque
import heapq

def build_graph(n, edges, directed=False):
    adj = {v: [] for v in range(n)}
    for e in edges:
        u, v = e[0], e[1]
        w = e[2] if len(e) > 2 else 1
        adj[u].append((v, w))
        if not directed:
            adj[v].append((u, w))
    return adj

def find_monitor(n, trust):
    # TODO: count in-degree and out-degree
    pass

In [ ]:
# Test cell: run after your solution
assert find_monitor(3, [(0, 2), (1, 2)]) == 2
assert find_monitor(3, [(0, 2), (1, 2), (2, 0)]) == -1
assert find_monitor(1, []) == 0
assert find_monitor(2, []) == -1
print("✅ Task 1 passed")

### Task 2 🟢 Is There a Route?
Given an undirected graph with n vertices and an edge list, return `True` if there is a path from `source` to `target`. Solve it with DFS using an explicit stack.

```text
n = 6, edges = [(0,1),(0,2),(3,5),(5,4),(4,3)], 0 → 5 → False
```

In [ ]:
def has_route(n, edges, source, target):
    adj = build_graph(n, edges)
    # TODO: iterative DFS
    pass

In [ ]:
# Test cell: run after your solution
assert has_route(6, [(0, 1), (0, 2), (3, 5), (5, 4), (4, 3)], 0, 5) is False
assert has_route(3, [(0, 1), (1, 2), (2, 0)], 0, 2) is True
assert has_route(1, [], 0, 0) is True
print("✅ Task 2 passed")

### Task 3 🟡 Shortest Route with BFS
Return the **actual shortest path** (list of vertices) from `start` to `goal` in an unweighted undirected graph, or `[]` if unreachable. Use BFS with a `parent` dictionary.

```text
edges [(0,1),(1,2),(0,3),(3,4),(4,2)], 0 → 2 → [0, 1, 2]
```

In [ ]:
def shortest_path(n, edges, start, goal):
    adj = build_graph(n, edges)
    parent = {start: None}
    # TODO: BFS, then rebuild the path from goal back to start
    return []

In [ ]:
# Test cell: run after your solution
assert shortest_path(5, [(0, 1), (1, 2), (0, 3), (3, 4), (4, 2)], 0, 2) == [0, 1, 2]
assert shortest_path(4, [(0, 1), (2, 3)], 0, 3) == []
assert shortest_path(3, [(0, 1)], 0, 0) == [0]
p = shortest_path(6, [(0, 1), (1, 2), (2, 5), (0, 3), (3, 4), (4, 5)], 0, 5)
assert len(p) == 4 and p[0] == 0 and p[-1] == 5
print("✅ Task 3 passed")

### Task 4 🟡 Count Flooded Regions (Connected Components on a Grid)
A flood map from satellite imagery is a grid of `"W"` (water) and `"."` (dry). Water cells connected up/down/left/right form one flooded region. Count the regions and return the **size of the largest** one as `(count, largest)`.

Use BFS from every unvisited water cell (each cell is a vertex; neighbours are adjacent cells).

```text
["WW..", "W..W", "..WW", "...."] → (2, 3)
```

In [ ]:
def flood_regions(grid):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    seen, count, largest = set(), 0, 0
    # TODO: BFS from each unvisited water cell; track region sizes
    return count, largest

In [ ]:
# Test cell: run after your solution
assert flood_regions(["WW..", "W..W", "..WW", "...."]) == (2, 3)
assert flood_regions(["...", "..."]) == (0, 0)
assert flood_regions(["W.W", ".W.", "W.W"]) == (5, 1)
assert flood_regions([]) == (0, 0)
print("✅ Task 4 passed")

### Task 5 🟡 Can I Finish My Degree? (Course Schedule)
There are `n` courses (0 … n−1) and prerequisite pairs `(a, b)` meaning **b must be taken before a**. Return a valid order of all courses, or `[]` if it is impossible because of a cycle.

Use Kahn’s topological sort. To make the answer deterministic, always take the **smallest** available course first (use a heap instead of a queue).

```text
n = 4, prereqs = [(1, 0), (2, 0), (3, 1), (3, 2)] → [0, 1, 2, 3]
```

In [ ]:
def course_order(n, prereqs):
    adj = {v: [] for v in range(n)}
    indeg = [0] * n
    # TODO: build the graph b -> a, then Kahn's algorithm with a min-heap
    return []

In [ ]:
# Test cell: run after your solution
assert course_order(4, [(1, 0), (2, 0), (3, 1), (3, 2)]) == [0, 1, 2, 3]
assert course_order(2, [(1, 0), (0, 1)]) == []
assert course_order(3, []) == [0, 1, 2]
assert course_order(3, [(0, 2), (1, 2)]) == [2, 0, 1]
print("✅ Task 5 passed")

### Task 6 🔴 Network Delay Time (Dijkstra)
A message is sent from server `k` in a network of `n` servers (1 … n). `times = [(u, v, w), ...]` means a directed link from u to v taking w milliseconds. Return the time for **all** servers to receive the message, or −1 if some cannot be reached.

Run Dijkstra from k; the answer is the maximum shortest distance.

```text
times = [(2,1,1), (2,3,1), (3,4,1)], n = 4, k = 2 → 2
```

In [ ]:
def network_delay(times, n, k):
    adj = {v: [] for v in range(1, n + 1)}
    for u, v, w in times:
        adj[u].append((v, w))
    # TODO: Dijkstra from k, then return the maximum distance (or -1)
    pass

In [ ]:
# Test cell: run after your solution
assert network_delay([(2, 1, 1), (2, 3, 1), (3, 4, 1)], 4, 2) == 2
assert network_delay([(1, 2, 1)], 2, 2) == -1
assert network_delay([(1, 2, 5), (1, 3, 1), (3, 2, 1)], 3, 1) == 2
assert network_delay([], 1, 1) == 0
print("✅ Task 6 passed")

### Task 7 🔴 Spread of Crop Disease (Multi-Source BFS)
A field is a grid: `0` empty, `1` healthy plant, `2` infected plant. Every day, each infected plant infects its healthy neighbours (up/down/left/right). Return the number of days until no healthy plant remains, or −1 if some plants can never be infected.

Start BFS from **all** infected plants at once (put them all in the queue with day 0).

```text
[[2,1,1],[1,1,0],[0,1,1]] → 4
```

In [ ]:
def days_to_infect(field):
    rows, cols = len(field), len(field[0]) if field else 0
    g = [row[:] for row in field]
    q, healthy = deque(), 0
    # TODO: enqueue all infected cells with day 0; BFS; count remaining healthy
    return -1

In [ ]:
# Test cell: run after your solution
assert days_to_infect([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert days_to_infect([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert days_to_infect([[0, 2]]) == 0
assert days_to_infect([[1, 1, 2, 1, 1]]) == 2
print("✅ Task 7 passed")

## 🔬 Bonus: Draw the Road Network with NetworkX

In [ ]:
# Bonus: run after completing the tasks
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    G = nx.Graph()
    for u in roads:
        for v, w in roads[u]:
            G.add_edge(u, v, weight=w)
    pos = nx.spring_layout(G, seed=7)
    nx.draw(G, pos, with_labels=True, node_color="#9fd3c7", node_size=1800, font_size=9)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "weight"))
    path = nx.dijkstra_path(G, "Sahiwal", "Sheikhupura")
    print("NetworkX shortest path:", path, nx.dijkstra_path_length(G, "Sahiwal", "Sheikhupura"), "km")
    plt.show()
except ImportError:
    print("networkx is not installed here; run this cell in Google Colab.")

---
## 🎉 Lab 16 Complete!

Next: **Course complete! See “Where to Go Next” in this manual.**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab16/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)